In [1]:
import numpy as np
import pandas as pd
import pickle
import tensorflow as tf
from tensorflow.keras.models import load_model

In [2]:
## Load the trained ANN model, scaler pickle, onhot
model = load_model('model.h5')

## load the encoder and scaler
with open('onehot_encoder_geo.pkl','rb') as file:
    onehot_encoder_geo = pickle.load(file)

with open('lable_encoder_gender.pkl','rb') as file:
    label_encoder_gender = pickle.load(file)

with open('scaler.pkl','rb') as file:
    scaler = pickle.load(file)

    


In [3]:
# Example input data
input_data = {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000
}

In [4]:
input_df=pd.DataFrame([input_data])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,Male,40,3,60000,2,1,1,50000


In [5]:
# Label encode Gender
input_df['Gender'] = label_encoder_gender.transform(input_df['Gender'])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,1,40,3,60000,2,1,1,50000


In [6]:

#  One‑hot encode Geography
geo_encoded = onehot_encoder_geo.transform(input_df[['Geography']]).toarray()
geo_encoded

array([[1., 0., 0.]])

In [7]:
geo_encoded_df = pd.DataFrame(geo_encoded, columns= onehot_encoder_geo.get_feature_names_out(['Geography']))
geo_encoded_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [8]:

# Combine all features
input_df = pd.concat(
    [input_df.drop(columns=['Geography']), geo_encoded_df],
    axis=1
)


In [9]:
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [10]:

 ## Scaling
input_scaled = scaler.transform(input_df)


In [11]:
input_data = pd.concat([input_df.reset_index(drop=True),geo_encoded_df], axis = 1)
input_data

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0,1.0,0.0,0.0


In [12]:

prediction = model.predict(input_scaled)
prediction_proba = prediction[0][0]


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 290ms/step


In [13]:

print("Churn Probability:", prediction_proba)
print("Churn Prediction:", "Yes" if prediction_proba > 0.5 else "No")



Churn Probability: 0.024710944
Churn Prediction: No
